# 00 — Scratch: Raw File Inspection

Open each source file independently and print shape, dtypes, head, tail, null counts, describe.
Purpose: validate what's actually inside each file before building loaders.

In [1]:
import sys, os
os.chdir(os.path.join(os.path.dirname(os.getcwd()), 'mari_poc') if 'mari_poc' not in os.getcwd() else os.getcwd())
sys.path.insert(0, '.')
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## 1. BHP.csv

In [3]:
df = pd.read_csv('../data/raw/BHP.csv')
print(f'Shape: {df.shape}')
print(f'Dtypes:\n{df.dtypes}')
print(f'\nHead(5):\n{df.head()}')
print(f'\nTail(5):\n{df.tail()}')
print(f'\nNull counts:\n{df.isnull().sum()}')
print(f'\nDescribe:\n{df.describe()}')

Shape: (315, 3)
Dtypes:
well     object
date     object
bhp     float64
dtype: object

Head(5):
  well       date     bhp
0   11   1-Nov-88  1053.0
1   11  19-May-90  1067.0
2   11  26-Nov-91  1048.0
3   11  11-Feb-92  1034.0
4   11  23-Aug-93  1022.0

Tail(5):
     well       date    bhp
310  122H  01-Jul-23  646.0
311  123H  01-Jul-24  601.0
312  124H  01-Jul-24  655.0
313  125H  01-May-25  560.0
314  126H  01-May-25    NaN

Null counts:
well    0
date    0
bhp     1
dtype: int64

Describe:
               bhp
count   314.000000
mean    855.523671
std     133.839755
min     512.414912
25%     762.000000
50%     867.000000
75%     961.527500
max    1153.200000


**Observations:**
- 315 rows × 3 cols (well, date, bhp)
- Well names are bare: '11', '122H', 'E-2' — no 'M-' prefix or '-HRL' suffix
- Dates are strings in mixed formats: '1-Nov-88', '01-Jul-23'
- 1 null BHP (M-126H, last entry)
- BHP range: 512–1153 psig

## 2. gas_gravity.csv

In [5]:
df = pd.read_csv('../data/raw/gas_gravity.csv')
print(f'Shape: {df.shape}')
print(f'Dtypes:\n{df.dtypes}')
print(f'\nFull content:')
print(df.to_string())
print(f'\nRaw lines:')
with open('../data/raw/gas_gravity.csv') as f:
    for line in f:
        print(repr(line))

Shape: (22, 2)
Dtypes:
well            object
gas_gravity    float64
dtype: object

Full content:
          well  gas_gravity
0     M-11-HRL         0.70
1   M-122H-HRL         0.77
2   M-123H-HRL         0.75
3   M-124H-HRL         0.76
4   M-125H-HRL         0.75
5   M-126H-HRL         0.78
6     M-13-HRL         0.69
7     M-22-HRL         0.77
8     M-41-HRL         0.69
9     M-50-HRL         0.76
10    M-51-HRL         0.77
11    M-56-HRL         0.72
12    M-57-HRL         0.75
13    M-58-HRL         0.76
14    M-61-HRL         0.80
15    M-63-HRL         0.75
16    M-65-HRL         0.72
17    M-67-HRL         0.76
18    M-75-HRL         0.74
19    M-81-HRL         0.77
20    M-82-HRL         0.77
21   M-E-2-HRL         0.78

Raw lines:
'well,gas_gravity\n'
'M-11-HRL,\t0.7\n'
'M-122H-HRL,\t0.77\n'
'M-123H-HRL,\t0.75\n'
'M-124H-HRL,\t0.76\n'
'M-125H-HRL,\t0.75\n'
'M-126H-HRL,\t0.78\n'
'M-13-HRL,\t0.69\n'
'M-22-HRL,\t0.77\n'
'M-41-HRL,\t0.69\n'
'M-50-HRL,\t0.76\n'
'M-51-HRL,\t0.77

**Observations:**
- 22 rows, well names already in canonical M-XX-HRL form
- Tab character before each numeric value (e.g. `\t0.7`)
- Pandas coerces correctly despite tabs
- Range: 0.685–0.803
- A second source of gas gravity exists in the Pressure xlsx with slightly more precise values (e.g. 0.769 vs 0.77)

## 3. Subsurface data — Stixors Technologies.xlsx

In [6]:
xl = pd.ExcelFile('../data/raw/Subsurface data- Stixors Technologies.xlsx')
print(f'Sheet names: {xl.sheet_names}')
df = xl.parse('Additional Data shared', header=2)
df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
print(f'\nShape: {df.shape}')
print(f'Dtypes:\n{df.dtypes}')
print(f'\nHead(5):\n{df.head()}')
print(f'\nTail(5):\n{df.tail()}')
print(f'\nNull counts:\n{df.isnull().sum()}')
print(f'\nDescribe:\n{df.describe()}')

Sheet names: ['Additional Data shared']

Shape: (22, 9)
Dtypes:
Wells                            object
Porosity                        float64
Permeability (mD) - PTA         float64
Skin                            float64
Water saturation                float64
Net pay (m) \nMeasured Depth    float64
Chlorides (ppm)                   int64
Top Perf (m)                    float64
Bottom Perf (m)                 float64
dtype: object

Head(5):
        Wells  Porosity  Permeability (mD) - PTA  Skin  Water saturation  \
0    M-11-HRL      0.20                      7.4  -0.1              0.45   
1  M-122H-HRL      0.22                     34.0   NaN              0.46   
2  M-123H-HRL      0.24                     22.5   NaN              0.30   
3  M-124H-HRL      0.22                     34.0   NaN              0.46   
4  M-125H-HRL      0.24                     22.5   NaN              0.30   

   Net pay (m) \nMeasured Depth  Chlorides (ppm)  Top Perf (m)  \
0                          10

**Observations:**
- Single sheet 'Additional Data shared', header on row 2 (rows 0-1 are blank)
- 22 rows × 9 meaningful columns (plus 2 unnamed + 1 'Notes:' column dropped)
- Well names already canonical
- 5 null Skin values — all horizontals (M-122H through M-126H)
- Net pay: verticals 6–15 m, horizontals 475–802 m — **different physical quantity** (completed lateral length vs net pay thickness)
- M-122H/M-124H/M-126H share identical (φ=0.22, k=34, Sw=0.46) — likely analog-copied, not measured
- M-123H/M-125H share (φ=0.24, k=22.5, Sw=0.30)
- Chlorides range: 1215–23000 ppm; M-123H has 1215 — suspiciously low vs the rest (≥7000)

## 4. STIXOR Sharing Data.xlsx — Rates sheet

In [7]:
df = pd.read_excel('../data/raw/STIXOR Sharing Data.xlsx', sheet_name='Rates', header=1)
df.columns = df.columns.str.strip()
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'Dtypes:\n{df.dtypes}')
print(f'\nHead(5):\n{df.head()}')
print(f'\nTail(5):\n{df.tail()}')
print(f'\nNull counts:\n{df.isnull().sum()}')
print(f'\nDescribe:\n{df.describe()}')

Shape: (6652, 10)
Columns: ['Unnamed: 0', 'Date', 'Wells', 'CHOKE 1/64"', 'DAYS on production', 'Monthly Gas Produced MMcf (volume)', 'Monthly Water Produced bbl (volume)', 'Unnamed: 7', 'Notes', 'Unnamed: 9']
Dtypes:
Unnamed: 0                                    float64
Date                                   datetime64[ns]
Wells                                          object
CHOKE 1/64"                                   float64
DAYS on production                            float64
Monthly Gas Produced MMcf (volume)            float64
Monthly Water Produced bbl (volume)           float64
Unnamed: 7                                    float64
Notes                                          object
Unnamed: 9                                     object
dtype: object

Head(5):
   Unnamed: 0       Date     Wells  CHOKE 1/64"  DAYS on production  \
0         NaN 1978-03-01  M-11-HRL          NaN                 NaN   
1         NaN 1978-04-01  M-11-HRL          NaN                 NaN   
2    

**Observations:**
- Header on row 1 (row 0 is blank). Column names have heavy leading whitespace.
- 6652 rows × 10 cols (after dropping unnamed/notes)
- 22 wells, dates 1978-03 to 2025-06
- Dates are proper timestamps (no Excel serial number issue here)
- gas_mmcf: 30 nulls; water_bbl: 1969 nulls (many months are gas-only)
- choke_64ths and prod_days: ~4560 nulls (early production lacks these)
- 'Notes' column contains metadata text in first few rows, then NaN

## 5. STIXOR Sharing Data.xlsx — Pressures sheet

In [8]:
df = pd.read_excel('../data/raw/STIXOR Sharing Data.xlsx', sheet_name='Pressures', header=1)
df.columns = df.columns.str.strip()
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'Dtypes:\n{df.dtypes}')
print(f'\nHead(5):\n{df.head()}')
print(f'\nTail(5):\n{df.tail()}')
print(f'\nNull counts:\n{df.isnull().sum()}')
print(f'\nDescribe:\n{df.describe()}')

Shape: (5354, 5)
Columns: ['Unnamed: 0', 'Date', 'Wells', 'Wellhead Flowing Pressure (WHFP)psig', 'Line Pressure psig']
Dtypes:
Unnamed: 0                                     float64
Date                                    datetime64[ns]
Wells                                           object
Wellhead Flowing Pressure (WHFP)psig           float64
Line Pressure psig                             float64
dtype: object

Head(5):
   Unnamed: 0       Date     Wells  Wellhead Flowing Pressure (WHFP)psig  \
0         NaN 1990-06-01  M-11-HRL                                 793.1   
1         NaN 1990-08-01  M-11-HRL                                   NaN   
2         NaN 1990-09-01  M-11-HRL                                 829.3   
3         NaN 1990-10-01  M-11-HRL                                   NaN   
4         NaN 1995-01-01  M-11-HRL                                 818.5   

   Line Pressure psig  
0                 NaN  
1                 NaN  
2                 NaN  
3                 Na

**Observations:**
- 5354 rows × 5 cols, header on row 1
- 22 wells, dates 1990-06 to 2025-06
- WHFP: 242 nulls; Line Pressure: 3339 nulls (line pressure very sparse pre-2005)
- WHFP range: ~60–1100 psig; Line Pressure: ~110–400 psig

## 6. Pressure data — Stixors Technologies.xlsx

In [9]:
xl = pd.ExcelFile('../data/raw/Pressure data- Stixors Technologies.xlsx')
print(f'Sheet names: {xl.sheet_names}')
df = xl.parse('Sheet2', header=None)
print(f'\nSheet2 shape: {df.shape}')
print(f'Row 2 (headers): {list(df.iloc[2])}')

# Parse the three BHP groups
for name, cols in [('Group1 (cols 2-4)', (2,3,4)), ('Group2 (cols 6-8)', (6,7,8)), ('Group3 (cols 10-12)', (10,11,12))]:
    g = df.iloc[3:, list(cols)].dropna(how='all')
    g.columns = ['well','date','pressure']
    g = g.dropna(subset=['pressure'])
    print(f'\n{name}: {len(g)} rows, wells: {sorted(g["well"].astype(str).str.strip().unique())}')

# Gas gravity section
gg = df.iloc[3:, [14,15]].dropna(how='all')
gg.columns = ['well','gas_gravity']
print(f'\nGas gravity section: {len(gg)} rows')
print(gg.to_string())

# GWC note
print(f'\nGWC Note (col 17): {df.iloc[3, 17]}')

Sheet names: ['Sheet2', 'Well Completion Sketch ']

Sheet2 shape: (138, 18)
Row 2 (headers): [nan, nan, 'Well # ', 'Date ', 'Pressure', nan, 'Well # ', 'Date ', 'Pressure', nan, 'Well # ', 'Date ', 'Pressure', nan, 'Wells', 'Gas Specific Gravity', nan, 'Note:']

Group1 (cols 2-4): 122 rows, wells: ['11', '13', '22', '41', '50']

Group2 (cols 6-8): 135 rows, wells: ['51', '56', '57', '58', '61', '63', '65']

Group3 (cols 10-12): 57 rows, wells: ['122H', '123H', '124H', '125H', '67', '75', '81', '82', 'E-2']

Gas gravity section: 22 rows
          well gas_gravity
3     M-11-HRL         0.7
4   M-122H-HRL       0.769
5   M-123H-HRL       0.748
6   M-124H-HRL       0.761
7   M-125H-HRL        0.75
8   M-126H-HRL        0.78
9     M-13-HRL       0.689
10    M-22-HRL       0.766
11    M-41-HRL       0.685
12    M-50-HRL    0.758554
13    M-51-HRL       0.765
14    M-56-HRL       0.719
15    M-57-HRL       0.751
16    M-58-HRL       0.755
17    M-61-HRL       0.803
18    M-63-HRL       0.745

**Observations:**
- Two sheets: 'Sheet2' (BHP + gas gravity + GWC note) and 'Well Completion Sketch' (images, not tabular data)
- Sheet2 has a multi-column layout: three side-by-side (Well#, Date, Pressure) groups
- 314 BHP records across 21 wells — M-126H is absent (BHP.csv has it with NaN BHP)
- Gas gravity values here are more precise than gas_gravity.csv (e.g. 0.769 vs 0.77)
- **GWC note confirms: 'The GWC for the entire reservoir is 684 m TVD SS'**
- Well Completion Sketch sheet contains embedded images (M-122H and M-124H), not parseable as data

**Decision:** Use BHP.csv as the primary BHP source (it has all 22 wells). Use gas_gravity.csv for gas gravity (rounded but complete). The Pressure xlsx serves as a cross-check.